# Phase 11 — Optional Transfer Learning Experiment

In this notebook, we experiment with Transfer Learning using a pre-trained ResNet18. 
We will adapt our 28x28 grayscale images to the format expected by ImageNet models (3 channels, 224x224), freeze the base layers, fine-tune a new classifier head, and then compare the results against our Baseline and Small CNN models.

In [1]:
# ── Step 0: Imports ───────────────────────────────────────────────────────────
import os
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
import medmnist
from medmnist import INFO

DATA_CACHE_DIR = '../data/'
SPLIT_MANIFEST_PATH = '../data/processed/split_manifest.csv'
RESULTS_DIR = '../results/'
MODEL_DIR = '../models/'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [2]:
# ── Step 1: Prepare Dataset & Dataloaders for ResNet ──────────────────────────
manifest = pd.read_csv(SPLIT_MANIFEST_PATH)
info = INFO['pneumoniamnist']
DataClass = getattr(medmnist, info['python_class'])
dataset = DataClass(split='train', download=False, root=DATA_CACHE_DIR)

train_idx = manifest[manifest['split'] == 'train']['original_index'].values
val_idx = manifest[manifest['split'] == 'val']['original_index'].values

X_train_raw, y_train = dataset.imgs[train_idx], dataset.labels.squeeze()[train_idx]
X_val_raw, y_val = dataset.imgs[val_idx], dataset.labels.squeeze()[val_idx]

# ImageNet requires 3 channels (RGB) and expects larger resolutions (e.g. 224x224).
transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)), # Convert grayscale to RGB
    transforms.Resize((224, 224), antialias=True),
    transforms.RandomRotation(degrees=10),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_eval = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)),
    transforms.Resize((224, 224), antialias=True),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class MedMNISTSubset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        from PIL import Image
        img = Image.fromarray(self.images[idx])
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor([label], dtype=torch.float32)

train_loader = DataLoader(MedMNISTSubset(X_train_raw, y_train, transform=transform_train), batch_size=16, shuffle=True)
val_loader = DataLoader(MedMNISTSubset(X_val_raw, y_val, transform=transform_eval), batch_size=16, shuffle=False)
print("Dataloaders ready for ImageNet-sized inputs.")

Dataloaders ready for ImageNet-sized inputs.


In [3]:
# ── Step 2: Initialize Pre-trained ResNet18 & Modify Head ─────────────────────
# Load pre-trained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze the base layers
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully connected layer (which naturally has requires_grad=True)
# Binary classification requires an output size of 1
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 1)

print("ResNet18 model loaded with frozen base and new randomly initialized classification head.")

ResNet18 model loaded with frozen base and new randomly initialized classification head.


In [4]:
# ── Step 3: Train the Transfer Learning Model ─────────────────────────────────
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001) # Only optimize the new FC layer
EPOCHS = 10  # Pre-trained features converge quickly on the classifier

start_time = time.time()
print("\n=== Training ResNet18 (Feature Extraction) ===")

device = torch.device('cpu') # Running locally on CPU for education
model.to(device)

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)
        
    train_acc = correct_train / total_train
    
    model.eval()
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            correct_val += (preds == labels).sum().item()
            total_val += labels.size(0)
            
    val_acc = correct_val / total_val
    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {train_loss/total_train:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

runtime = time.time() - start_time

# Save Metrics
transfer_metrics = {
    "model_name": "ResNet18_TransferLearning",
    "hyperparameters": {"epochs": EPOCHS, "batch_size": 16, "lr": 0.001, "frozen_base": True},
    "train_metric": train_acc,
    "validation_metric": val_acc,
    "runtime_seconds": round(runtime, 2),
    "seed": SEED
}

with open(os.path.join(RESULTS_DIR, 'transfer_metrics.json'), 'w') as f:
    json.dump(transfer_metrics, f, indent=4)

MODEL_PATH = os.path.join(MODEL_DIR, 'pneumonia_resnet18_v1.pt')
torch.save(model.state_dict(), MODEL_PATH)

print(f"\n✅ Transfer Learning Complete! Runtime: {runtime:.2f}s")
print(f"✅ Model metrics saved to transfer_metrics.json and weights to {MODEL_PATH}")


=== Training ResNet18 (Feature Extraction) ===


Epoch [1/10] Loss: 0.7120 | Train Acc: 0.4143 | Val Acc: 0.4667


Epoch [2/10] Loss: 0.6508 | Train Acc: 0.6429 | Val Acc: 0.4667


Epoch [3/10] Loss: 0.5778 | Train Acc: 0.8143 | Val Acc: 0.4000


Epoch [4/10] Loss: 0.5447 | Train Acc: 0.7714 | Val Acc: 0.4667


...

Epoch [7/10] Loss: 0.4297 | Train Acc: 0.8857 | Val Acc: 0.6000


Epoch [8/10] Loss: 0.4351 | Train Acc: 0.8143 | Val Acc: 0.6000


Epoch [9/10] Loss: 0.4140 | Train Acc: 0.8429 | Val Acc: 0.6000


Epoch [10/10] Loss: 0.3883 | Train Acc: 0.8857 | Val Acc: 0.6000



✅ Transfer Learning Complete! Runtime: 21.26s
✅ Model metrics saved to transfer_metrics.json and weights to ../models/pneumonia_resnet18_v1.pt


In [5]:
# ── Step 4: Compare Baseline vs Small CNN vs Transfer Learning ────────────────
import json
import os

models = ['baseline_metrics.json', 'cnn_metrics.json', 'transfer_metrics.json']
results = []
for mf in models:
    path = os.path.join(RESULTS_DIR, mf)
    if os.path.exists(path):
        with open(path, 'r') as f:
            data = json.load(f)
            results.append({
                'Model': data['model_name'],
                'Train Acc': f"{data['train_metric']:.3f}",
                'Val Acc': f"{data['validation_metric']:.3f}",
                'Runtime (s)': f"{data['runtime_seconds']:.2f}"
            })

comparison_df = pd.DataFrame(results)
print("\n=== Model Comparison ===")
print(comparison_df.to_markdown(index=False))


=== Model Comparison ===
| Model                       |   Train Acc |   Val Acc |   Runtime (s) |
|:----------------------------|------------:|----------:|--------------:|
| LogisticRegression_Baseline |       1     |     0.8   |          0.02 |
| TinyCNN                     |       0.986 |     0.933 |          1.21 |
| ResNet18_TransferLearning   |       0.886 |     0.6   |         21.26 |
